# The CPython Cost Model

Big-O tells you how an algorithm **scales**; it says nothing about the **constant factor** hiding inside each step. In CPython those constants are enormous — every iteration of a pure-Python loop pays for bytecode dispatch, boxed integer objects, and reference-count bookkeeping, with no JIT to erase it. This notebook makes those constants concrete: where the time actually goes, why two O(n) loops can differ by 10x or more, and the handful of moves (builtins, comprehensions, numpy, hoisting lookups) that buy back the constant. It ties together the internals thesis running through the whole series. Every number below is measured at run time, so re-execute and expect the exact figures to drift — what's stable is the **ratios**.

**Contents**

1. **Big-O isn't the whole story** — two O(n) loops, wildly different real cost
2. **Why pure-Python loops are slow** — bytecode dispatch, boxed ints, no JIT
3. **Function-call overhead** — every call builds a frame
4. **Attribute & global lookup** — `LOAD_GLOBAL`/`LOAD_ATTR` vs `LOAD_FAST`
5. **When to drop to C** — builtins, comprehensions, numpy vectorization
6. **The GIL** — threads, CPU-bound vs IO-bound, free-threading
7. **Cost-model cheat-sheet**

## 1. Big-O isn't the whole story

The **big-o** notebook ranks algorithms by growth rate, and that ranking is the right first question: an O(n) approach crushes an O(n²) one once n is large. But two algorithms with the *same* Big-O can still differ by an order of magnitude in wall-clock time, because Big-O deliberately throws away the constant factor — and in CPython that constant is where the action is.

Below, both loops are O(n): each sums the integers `0..n-1`. The only difference is that one does its work in a pure-Python `for` loop and the other hands the same work to the built-in `sum`, which runs in C. Same asymptotics, very different reality:

In [1]:
import timeit

n = 1_000_000
data = list(range(n))

def python_loop(xs):
    total = 0
    for x in xs:          # O(n) — but every step is interpreted bytecode
        total += x
    return total

t_loop    = timeit.timeit(lambda: python_loop(data), number=5) / 5
t_builtin = timeit.timeit(lambda: sum(data),         number=5) / 5

assert python_loop(data) == sum(data)        # identical result, identical Big-O
print(f"python for-loop : {t_loop*1000:7.2f} ms")
print(f"builtin sum()   : {t_builtin*1000:7.2f} ms")
print(f"same O(n), but the loop is ~{t_loop/t_builtin:.0f}x slower")

python for-loop :   10.71 ms
builtin sum()   :    3.60 ms
same O(n), but the loop is ~3x slower


Same complexity class, same answer, and yet a several-fold gap. Big-O told you neither loop will fall off a cliff as n grows; it did **not** tell you which to write. The rest of this notebook is about that missing constant — what each Python step actually costs, and how to pay less.

## 2. Why pure-Python loops are slow

Three compounding reasons, none of which a faster CPU fixes on its own:

- **Bytecode dispatch.** CPython compiles your loop body to bytecode and the evaluation loop fetches, decodes, and dispatches each instruction one at a time. A trivial `total += x` is several bytecodes, and you pay that dispatch overhead *every iteration*.
- **Boxed integers.** A Python `int` isn't a machine word — it's a heap-allocated object with a refcount, a type pointer, and digit storage (28 bytes for a small int). Arithmetic unboxes operands, does the math, allocates a result object, and adjusts refcounts.
- **No JIT.** Through 3.14 the reference interpreter executes bytecode directly. It *specializes* hot bytecodes adaptively (PEP 659), which helps, but there's no machine-code compilation of your loop the way a JIT (PyPy) or a vectorized C kernel (numpy) gives you.

First, look at what one iteration actually compiles to:

In [2]:
import dis

def add_loop(xs):
    total = 0
    for x in xs:
        total += x
    return total

dis.dis(add_loop)

  3           RESUME                   0

  4           LOAD_SMALL_INT           0
              STORE_FAST               1 (total)

  5           LOAD_FAST_BORROW         0 (xs)
              GET_ITER
      L1:     FOR_ITER                11 (to L2)
              STORE_FAST               2 (x)

  6           LOAD_FAST_BORROW_LOAD_FAST_BORROW 18 (total, x)
              BINARY_OP               13 (+=)
              STORE_FAST               1 (total)
              JUMP_BACKWARD           13 (to L1)

  5   L2:     END_FOR
              POP_ITER

  7           LOAD_FAST_BORROW         1 (total)
              RETURN_VALUE


Every trip round the loop runs `FOR_ITER`, a `STORE_FAST`, the `BINARY_OP` for `+=`, another store, and a `JUMP_BACKWARD` — and behind `BINARY_OP` sits the unbox/add/rebox/refcount dance on boxed `int` objects. Now measure the payoff of moving that work into C three different ways: the built-in `sum`, and numpy's vectorized `ndarray.sum` (one C loop over a flat C-`int64` buffer, no per-element Python objects at all):

In [3]:
import timeit, numpy as np

n = 1_000_000
data = list(range(n))
arr  = np.arange(n)              # flat int64 buffer, not 1M boxed Python ints

def python_loop(xs):
    total = 0
    for x in xs:
        total += x
    return total

t_loop    = timeit.timeit(lambda: python_loop(data), number=5) / 5
t_builtin = timeit.timeit(lambda: sum(data),         number=5) / 5
t_numpy   = timeit.timeit(lambda: arr.sum(),         number=5) / 5

print(f"{'method':16}{'time (ms)':>11}{'vs loop':>12}")
for name, t in [("for-loop", t_loop), ("sum()", t_builtin), ("numpy .sum()", t_numpy)]:
    print(f"{name:16}{t*1000:>11.2f}{t_loop/t:>11.1f}x")

method            time (ms)     vs loop
for-loop              11.32        1.0x
sum()                  2.97        3.8x
numpy .sum()           0.24       47.4x


The boxed-int tax also shows up in **memory**: a Python list of n distinct ints stores n pointers *plus* n separate 28-byte int objects, while a numpy `int64` array is one contiguous 8-bytes-per-element buffer. That's the same density story the **arrays** notebook tells — Python's object-per-element model is flexible but heavy.

In [4]:
import sys, numpy as np

n = 1000
lst = list(range(1000, 1000 + n))      # ints above the small-int cache: distinct objects
arr = np.arange(1000, 1000 + n, dtype=np.int64)

container = sys.getsizeof(lst)
int_objs  = sum(sys.getsizeof(x) for x in lst)   # the boxed ints the pointers reference
list_total = container + int_objs

print(f"list: {container} B container + {int_objs} B int objects = {list_total} B")
print(f"numpy int64 array: {arr.nbytes} B data ({sys.getsizeof(arr)} B incl. header)")
print(f"list uses ~{list_total/arr.nbytes:.1f}x the memory for the same 1000 integers")

list: 8056 B container + 28000 B int objects = 36056 B
numpy int64 array: 8000 B data (8112 B incl. header)
list uses ~4.5x the memory for the same 1000 integers


## 3. Function-call overhead

A Python function call is not free: the interpreter builds a **frame** object (its own local variables, evaluation stack, and bookkeeping), binds the arguments, runs the body, then tears the frame down and returns. For a heavy function that's noise. For a *trivial* helper called in a hot loop, the frame setup can cost as much as the work itself.

Same computation, once inlined into the loop and once factored into a one-line helper called every iteration. The gap is the call overhead:

In [5]:
import timeit

N = 1_000_000

def inline():
    total = 0
    for i in range(N):
        total += i * i          # work done in place
    return total

def square(x):
    return x * x

def via_helper():
    total = 0
    for i in range(N):
        total += square(i)      # same work, one extra call per iteration
    return total

assert inline() == via_helper()
t_in = timeit.timeit(inline,     number=5) / 5
t_hp = timeit.timeit(via_helper, number=5) / 5

print(f"inline       : {t_in*1000:7.2f} ms")
print(f"via helper   : {t_hp*1000:7.2f} ms   ({t_hp/t_in:.2f}x slower)")
print(f"overhead per call ~ {(t_hp - t_in)/N*1e9:.1f} ns")

inline       :   20.66 ms
via helper   :   26.55 ms   (1.29x slower)
overhead per call ~ 5.9 ns


A few nanoseconds per call sounds trivial — and it is, until you make millions of them in an inner loop. The takeaway isn't *avoid functions* (they're how you write maintainable code); it's that in a genuinely hot loop, **inlining the body** or pushing the whole loop into a C builtin/comprehension removes a real cost. Recursion compounds this: every recursive call is another frame, which is part of why the **recursion-and-backtracking** notebook favors iterative reformulations for deep problems.

## 4. Attribute & global lookup overhead

Not all name lookups cost the same. A **local** variable is read with `LOAD_FAST` — an index into the frame's array of locals, essentially free. A **global** or builtin needs `LOAD_GLOBAL`, which probes the module dict and falls back to builtins. An **attribute/method** access needs `LOAD_ATTR` (a dict lookup on the object/type) every single time.

Hence the classic optimization: **hoist a global or bound method into a local before a hot loop** — e.g. `append = lst.append`, then call `append(x)` inside. The bytecode shows exactly what you save:

In [6]:
import dis

def cold(out):
    for i in range(3):
        out.append(i)        # LOAD_ATTR 'append' EVERY iteration

def hoisted(out):
    append = out.append      # one LOAD_ATTR, stored to a local
    for i in range(3):
        append(i)            # then just LOAD_FAST each iteration

print("=== cold: out.append(i) ===")
dis.dis(cold)
print("\n=== hoisted: append = out.append ===")
dis.dis(hoisted)

=== cold: out.append(i) ===
  3           RESUME                   0

  4           LOAD_GLOBAL              1 (range + NULL)
              LOAD_SMALL_INT           3
              CALL                     1
              GET_ITER
      L1:     FOR_ITER                20 (to L2)
              STORE_FAST               1 (i)

  5           LOAD_FAST_BORROW         0 (out)
              LOAD_ATTR                3 (append + NULL|self)
              LOAD_FAST_BORROW         1 (i)
              CALL                     1
              POP_TOP
              JUMP_BACKWARD           22 (to L1)

  4   L2:     END_FOR
              POP_ITER
              LOAD_CONST               1 (None)
              RETURN_VALUE

=== hoisted: append = out.append ===
  7           RESUME                   0

  8           LOAD_FAST_BORROW         0 (out)
              LOAD_ATTR                0 (append)
              STORE_FAST               1 (append)

  9           LOAD_GLOBAL              3 (range + NULL)
   

In the cold version each iteration runs `LOAD_FAST out` then `LOAD_ATTR append`; the hoisted version does the `LOAD_ATTR` once and then only `LOAD_FAST append` per iteration. Measure both the method-hoist and a global/attribute hoist (`math.sqrt`):

In [7]:
import timeit, math

N = 1_000_000

def append_cold():
    out = []
    for i in range(N):
        out.append(i)
    return out

def append_hoisted():
    out = []
    append = out.append          # bind the method to a local once
    for i in range(N):
        append(i)
    return out

def sqrt_global():
    total = 0.0
    for i in range(N):
        total += math.sqrt(i)    # LOAD_GLOBAL math + LOAD_ATTR sqrt each iter
    return total

def sqrt_local():
    total = 0.0
    sqrt = math.sqrt             # hoist to a local once
    for i in range(N):
        total += sqrt(i)
    return total

tc = timeit.timeit(append_cold,    number=5) / 5
th = timeit.timeit(append_hoisted, number=5) / 5
tg = timeit.timeit(sqrt_global,    number=5) / 5
tl = timeit.timeit(sqrt_local,     number=5) / 5

print(f"out.append(i)       : {tc*1000:7.2f} ms")
print(f"append = out.append : {th*1000:7.2f} ms   ({tc/th:.2f}x)")
print(f"math.sqrt(i)        : {tg*1000:7.2f} ms")
print(f"sqrt = math.sqrt    : {tl*1000:7.2f} ms   ({tg/tl:.2f}x)")

out.append(i)       :   20.50 ms
append = out.append :   19.70 ms   (1.04x)
math.sqrt(i)        :   24.19 ms
sqrt = math.sqrt    :   21.72 ms   (1.11x)


The hoist still wins, but notice the speedup is **modest** here. That's modern CPython: the adaptive specializing interpreter (PEP 659) rewrites a repeatedly-executed `LOAD_ATTR` into a cached fast path, quietly absorbing much of what this trick used to buy on older versions. Lesson: hoisting is real and harmless, but **measure** — on 3.14 it's a small constant, not a game-changer. The big wins come from dropping the Python loop entirely, which is the next section.

## 5. When to drop to C

Every fix so far shaves the constant; the real leap is **eliminating the per-element Python step**. The ladder, roughly fastest-to-slowest for bulk numeric/aggregate work:

1. **numpy vectorization** — one C loop over a flat typed buffer, zero Python objects per element.
2. **Builtins & comprehensions** — `sum`, `map`, `any`/`all`, and list/set/dict comprehensions run their iteration in C; a comprehension also skips the per-iteration `LOAD_ATTR`+`CALL` that an `.append` loop pays (section 4).
3. **A pure-Python loop** — maximum flexibility, maximum constant factor.

But numpy isn't free: building the array and crossing the Python↔C boundary has **fixed overhead**, so for tiny n a plain Python loop wins. Watch where numpy overtakes as n grows — the **break-even**:

In [8]:
import timeit, numpy as np

print(f"{'n':>8}{'py loop (ms)':>15}{'numpy (ms)':>13}{'winner':>9}")
for n in [10, 100, 1_000, 10_000, 100_000]:
    xs  = list(range(n))
    arr = np.arange(n)
    reps = max(5, 200_000 // n)
    t_py = timeit.timeit(lambda: sum(x * x for x in xs),  number=reps) / reps
    t_np = timeit.timeit(lambda: int((arr * arr).sum()), number=reps) / reps
    winner = "numpy" if t_np < t_py else "python"
    print(f"{n:>8}{t_py*1000:>15.4f}{t_np*1000:>13.4f}{winner:>9}")

       n   py loop (ms)   numpy (ms)   winner
      10         0.0002       0.0007   python
     100         0.0013       0.0006    numpy
    1000         0.0122       0.0010    numpy
   10000         0.1236       0.0032    numpy
  100000         1.5551       0.0294    numpy


At n=10 the Python loop wins — numpy's setup cost dominates — but by n=100 vectorization is already ahead, and the gap widens fast. Rule of thumb: **vectorize bulk numeric work; don't reach for numpy to process a handful of items.**

The same “stay in C” principle drives the **strings** notebook's headline result: building a big string with `s += piece` in a loop risks O(n²) copying, while collecting pieces and doing one `"".join(...)` keeps the whole concatenation inside a single C routine. One quick confirmation:

In [9]:
import timeit

N = 50_000

def build_plus():
    s = ""
    ref = []
    for _ in range(N):
        ref.append(s)        # pin a 2nd reference so += can't do its in-place trick
        s += "x"
    return s

def build_join():
    parts = []
    for _ in range(N):
        parts.append("x")
    return "".join(parts)    # one C-level concatenation

assert build_plus() == build_join()
t_plus = timeit.timeit(build_plus, number=3) / 3
t_join = timeit.timeit(build_join, number=3) / 3
print(f"+= in a loop (pinned) : {t_plus*1000:7.2f} ms")
print(f"\'\'.join(parts)       : {t_join*1000:7.2f} ms   ({t_plus/t_join:.0f}x faster)")

+= in a loop (pinned) :  520.26 ms
''.join(parts)       :    0.43 ms   (1203x faster)


## 6. The GIL

CPython protects its internals — refcounts, object state — with a single **Global Interpreter Lock**. At any instant, in a standard build, **only one thread executes Python bytecode**. The consequence for performance:

- **CPU-bound pure-Python work does not parallelize across threads.** Four threads each spinning in a Python loop take turns holding the GIL; you get roughly single-thread throughput (often *worse*, from lock hand-off overhead).
- **IO-bound work overlaps fine.** Blocking calls — `time.sleep`, socket/file reads — **release the GIL** while they wait, so other threads run. Threads are great for concurrency, just not CPU parallelism.

First confirm what runtime we're on, then prove both halves:

In [10]:
import sys

# sys._is_gil_enabled() exists on 3.13+; True on a standard build, False on free-threaded
gil_on = sys._is_gil_enabled() if hasattr(sys, "_is_gil_enabled") else True
print(f"Python              : {sys.version.split()[0]}")
print(f"sys._is_gil_enabled : {gil_on}")
print(f"sys.flags.gil       : {getattr(sys.flags, "gil", "n/a")}   (1 = GIL on, 0 = free-threaded)")

Python              : 3.14.3
sys._is_gil_enabled : True
sys.flags.gil       : 1   (1 = GIL on, 0 = free-threaded)


In [11]:
import time
from concurrent.futures import ThreadPoolExecutor

def cpu_work(n):
    total = 0
    for i in range(n):       # pure-Python: holds the GIL the whole time
        total += i * i
    return total

tasks = [3_000_000] * 4

t0 = time.perf_counter()
for n in tasks:
    cpu_work(n)
serial = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(cpu_work, tasks))
threaded = time.perf_counter() - t0

print(f"CPU-bound, serial    : {serial*1000:7.1f} ms")
print(f"CPU-bound, 4 threads : {threaded*1000:7.1f} ms   (speedup {serial/threaded:.2f}x — ~none)")

CPU-bound, serial    :   232.8 ms
CPU-bound, 4 threads :   291.9 ms   (speedup 0.80x — ~none)


In [12]:
import time
from concurrent.futures import ThreadPoolExecutor

def io_work(d):
    time.sleep(d)            # blocks but RELEASES the GIL while waiting
    return d

tasks = [0.05] * 4

t0 = time.perf_counter()
for d in tasks:
    io_work(d)
serial = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(io_work, tasks))
threaded = time.perf_counter() - t0

print(f"IO-bound, serial     : {serial*1000:7.1f} ms")
print(f"IO-bound, 4 threads  : {threaded*1000:7.1f} ms   (speedup {serial/threaded:.2f}x — real overlap)")

IO-bound, serial     :   200.8 ms
IO-bound, 4 threads  :    52.3 ms   (speedup 3.84x — real overlap)


So for **CPU-bound** pure-Python work, threads won't help. The escape hatches:

- **`multiprocessing` / `ProcessPoolExecutor`** — separate processes, each with its own interpreter and GIL, give true parallelism (at the cost of process startup and inter-process data copying).
- **Drop to C** — numpy and many C extensions release the GIL during heavy computation, so a vectorized call can use multiple cores even from threads.
- **Free-threaded CPython** — 3.13 introduced an *experimental* build (PEP 703) that can run with the GIL disabled, and the work continues through 3.14. The cells above detect it: on a standard build `sys._is_gil_enabled()` is `True` and `sys.flags.gil` is `1`; on a free-threaded build they'd report `False`/`0` and the CPU-bound demo could actually scale across threads. It is still experimental and not the default, so the GIL remains the right mental model for everyday CPython.

## 7. Cost-model cheat-sheet

Relative costs are rough orders of magnitude on CPython 3.14 — re-run the cells for live numbers. The through-line: **the less time your data spends as per-element Python objects in an interpreted loop, the faster you go.**

| Pattern / operation | Rough relative cost | Faster idiom |
|---|:---:|---|
| `for`-loop accumulate over a list | baseline | `sum()` / comprehension / numpy |
| Built-in over an iterable (`sum`, `max`, `any`) | ~3–5x cheaper than the loop | already optimal |
| numpy vectorized op (large n) | ~10x+ cheaper than the loop | already optimal |
| numpy on tiny n (≲ ~50 elems) | *slower* (setup overhead) | plain Python loop |
| Trivial helper called in a hot loop | + a few ns per call | inline it / push loop into C |
| Deep recursion | a frame per call | iterate (see **recursion-and-backtracking**) |
| `obj.method(x)` in a hot loop | extra `LOAD_ATTR` each iter | hoist: `m = obj.method` (small win on 3.14) |
| `global_name` in a hot loop | `LOAD_GLOBAL` each iter | hoist to a local |
| Local variable (`LOAD_FAST`) | cheapest name access | — |
| `s += piece` building a big string | risks O(n²) | `"".join(parts)` (see **strings**) |
| `.append` in a loop to build a list | per-iter call + `LOAD_ATTR` | list comprehension |
| CPU-bound work across threads | ~no speedup (GIL) | `multiprocessing` / numpy / free-threaded build |
| IO-bound work across threads | real overlap | threads / `asyncio` |

And the meta-rule from section 1: **Big-O first** (the **big-o** notebook) to pick the algorithm class, then this cost model to shrink the constant within it.